# 🧠 GM-GReFEL: Dynamic Facial Expression Recognition Demo

**Paper:** *GM-GReFEL: A Geometry-Aware Spatiotemporal State-Space Architecture for Dynamic Facial Expression Recognition in the Wild*

**Authors:** Yudhistira Arditya Pratama, Yi-Zeng Hsieh (NTUST)

---

### ✅ Instructions:
1. Click **Runtime → Change runtime type → GPU (T4)**
2. Click **Runtime → Run all** (or press `Ctrl+F9`)
3. Wait ~3 minutes for setup to complete
4. Click the **public Gradio URL** that appears at the bottom (e.g. `https://xxxx.gradio.live`)

> Supports: **Static image** upload, **Video** upload, and **Webcam** snapshot — with GradCAM + face landmark visualization.

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

pkgs = [
    'gradio==4.44.0',
    'timm',
    'einops',
    'mediapipe',
    'flask-cors',
    'scipy',
    'opencv-python-headless',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('✅ Packages installed')

In [ ]:
# Cell 2 — Clone repository
import os, sys, subprocess

REPO_URL = 'https://github.com/ardiytama/GroupMamba-DFER.git'
REPO_DIR = '/content/GM-GReFEL'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print('✅ Repository ready at', REPO_DIR)

In [ ]:
# Cell 3 — Download model weights & MediaPipe
import os, urllib.request

WEIGHTS_URL  = 'https://github.com/ardiytama/GroupMamba-DFER/releases/download/v1.0-assets/best_model_fold_5.pth'
WEIGHTS_PATH = '/content/best_model_fold_5.pth'
MP_URL       = 'https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task'
MP_PATH      = '/tmp/face_landmarker.task'

if not os.path.exists(WEIGHTS_PATH):
    print('Downloading model weights (~1.2 GB)...')
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print('  Done.')

if not os.path.exists(MP_PATH):
    print('Downloading MediaPipe face landmarker...')
    urllib.request.urlretrieve(MP_URL, MP_PATH)
    print('  Done.')

print('✅ All weights ready')

In [ ]:
# Cell 4 — Load GM-GReFEL model
import torch, torch.nn.functional as F
import numpy as np, cv2, os
from PIL import Image
from torchvision import transforms
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python.vision import (
    FaceLandmarker, FaceLandmarkerOptions, RunningMode
)
from model import create_landmark_enhanced_efficientfer_ssm_dfew

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMOTIONS   = ['Happy', 'Sad', 'Neutral', 'Angry', 'Surprise', 'Disgust', 'Fear']
EMOJIS     = ['😄', '😢', '😐', '😠', '😲', '🤢', '😨']
NUM_FRAMES = 16

TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print(f'Loading GM-GReFEL on {DEVICE}...')
model = create_landmark_enhanced_efficientfer_ssm_dfew(num_classes=7).to(DEVICE)
ckpt  = torch.load(WEIGHTS_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt.get('model_state_dict', ckpt), strict=False)
model.eval()
for p in model.parameters():
    p.requires_grad = True

# SSM activation hook
_ssm_cache = []
def _ssm_hook(module, inp, out):
    _ssm_cache.clear()
    if isinstance(out, torch.Tensor):
        _ssm_cache.append(out.detach().cpu().float())
model.temporal.register_forward_hook(_ssm_hook)

# MediaPipe face landmarker
_mp_opts = FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=MP_PATH),
    running_mode=RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.3,
    min_face_presence_confidence=0.3,
    min_tracking_confidence=0.3,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)
LANDMARKER = FaceLandmarker.create_from_options(_mp_opts)
print(f'✅ GM-GReFEL ready on {DEVICE}')

In [ ]:
# Cell 5 — GradCAM + Landmark helpers
_LIPS  = [61,185,40,39,37,0,267,269,270,409,291,146,91,181,84,17,314,405,321,375,78,191,80,81,82,13,312,311,310,415,308,95,88,178,87,14,317,402,318,324,308]
_L_EYE = [33,7,163,144,145,153,154,155,133,33,246,161,160,159,158,157,173,133]
_R_EYE = [362,382,381,380,374,373,390,249,263,362,398,384,385,386,387,388,466,263]
_L_BROW= [70,63,105,66,107,55,65,52,53,46]
_R_BROW= [336,296,334,293,300,285,295,282,283,276]
_NOSE  = [168,6,197,195,5,4,1,19,94,2,98,97,2,326,327,2,98,240,99,60,327,460,289,305,94]
_OVAL  = [10,338,297,332,284,251,389,356,454,323,361,288,397,365,379,378,400,377,152,148,176,149,150,136,172,58,132,93,234,127,162,21,54,103,67,109,10]

def _draw_conn(img, lm, idxs, color, t=1):
    h, w = img.shape[:2]; n = len(lm)
    pts = [(int(lm[i].x*w), int(lm[i].y*h)) for i in idxs if i < n]
    for a, b in zip(pts, pts[1:]):
        cv2.line(img, a, b, color, t, cv2.LINE_AA)

def _draw_dots(img, lm, idxs, color, r=2):
    h, w = img.shape[:2]; n = len(lm)
    for i in idxs:
        if i < n:
            cv2.circle(img, (int(lm[i].x*w), int(lm[i].y*h)), r, color, -1)

def draw_landmarks(rgb):
    out = rgb.copy()
    res = LANDMARKER.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.face_landmarks:
        return out
    lm = res.face_landmarks[0]
    _draw_conn(out, lm, _OVAL,   (200,200,200))
    _draw_conn(out, lm, _L_BROW, (0,220,0));   _draw_conn(out, lm, _R_BROW, (0,220,0))
    _draw_conn(out, lm, _L_EYE,  (0,220,255)); _draw_conn(out, lm, _R_EYE,  (0,220,255))
    _draw_conn(out, lm, _NOSE,   (255,200,0))
    _draw_conn(out, lm, _LIPS,   (0,120,255))
    _draw_dots(out, lm, _OVAL,   (180,180,180))
    _draw_dots(out, lm, _L_EYE,  (0,220,255)); _draw_dots(out, lm, _R_EYE,  (0,220,255))
    _draw_dots(out, lm, _L_BROW, (0,255,0));   _draw_dots(out, lm, _R_BROW, (0,255,0))
    _draw_dots(out, lm, _NOSE,   (255,200,0))
    _draw_dots(out, lm, _LIPS,   (0,120,255))
    return out

def compute_gradcam(ft, pred_class):
    ft2 = ft.clone().detach().requires_grad_(True).to(DEVICE)
    model.zero_grad()
    model(ft2)[0, pred_class].backward()
    grads = ft2.grad.detach().cpu().numpy()[0]
    masks = np.mean(np.abs(grads), axis=1)
    for i in range(len(masks)):
        if masks[i].max() > 0:
            masks[i] = cv2.GaussianBlur(masks[i], (31,31), 0)
            masks[i] = (masks[i]-masks[i].min())/(masks[i].max()-masks[i].min()+1e-8)
    return masks

def overlay_heatmap(rgb, mask):
    h, w = rgb.shape[:2]
    m = cv2.resize(mask, (w, h))
    hm = cv2.applyColorMap(np.uint8(255*m), cv2.COLORMAP_JET)
    hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB).astype(np.float32)/255.
    return np.uint8(np.clip(0.55*hm + 0.45*rgb.astype(np.float32)/255., 0, 1)*255)

def run_inference(raw_frames):
    while len(raw_frames) < NUM_FRAMES:
        raw_frames = (raw_frames * 2)[:NUM_FRAMES]
    raw_frames = raw_frames[:NUM_FRAMES]
    tensors = [TRANSFORM(Image.fromarray(f)) for f in raw_frames]
    ft = torch.stack(tensors).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs = F.softmax(model(ft), dim=1)[0].cpu().numpy()
    pred  = int(probs.argmax())
    mid   = cv2.resize(raw_frames[NUM_FRAMES//2], (224,224))
    masks = compute_gradcam(ft, pred)
    cam   = overlay_heatmap(mid, masks[NUM_FRAMES//2])
    lm_img= draw_landmarks(mid)
    label = f"{EMOJIS[pred]}  {EMOTIONS[pred]}  ({probs[pred]*100:.1f}%)"
    probs_dict = {EMOTIONS[i]: float(f'{probs[i]:.4f}') for i in range(7)}
    return label, probs_dict, Image.fromarray(cam), Image.fromarray(lm_img)

print('✅ Inference helpers ready')

In [ ]:
# Cell 6 — Launch Gradio demo
import gradio as gr

def infer_image(pil_img):
    if pil_img is None:
        return 'No image provided', {}, None, None
    return run_inference([np.array(pil_img.convert('RGB'))])

def infer_video(video_path):
    if video_path is None:
        return 'No video provided', {}, None, None
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step  = max(1, total // NUM_FRAMES)
    frames = []
    for i in range(NUM_FRAMES):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i * step)
        ret, frm = cap.read()
        if not ret: break
        frames.append(cv2.cvtColor(frm, cv2.COLOR_BGR2RGB))
    cap.release()
    if not frames:
        return 'Could not read video', {}, None, None
    return run_inference(frames)

with gr.Blocks(
    title='GM-GReFEL Demo',
    theme=gr.themes.Soft(primary_hue='indigo', secondary_hue='purple'),
) as demo:

    gr.HTML("""
    <div style='text-align:center; padding:20px 0'>
      <h1 style='font-size:2em; font-weight:800; color:#3730a3'>🧠 GM-GReFEL</h1>
      <p>A Geometry-Aware Spatiotemporal State-Space Architecture for
         Dynamic Facial Expression Recognition in the Wild</p>
      <p style='font-size:0.85em; color:#888'>
         Yudhistira Arditya Pratama &amp; Yi-Zeng Hsieh &nbsp;·&nbsp; NTUST &nbsp;·&nbsp;
         IEEE Transactions on Image Processing (under review)
      </p>
    </div>
    """)

    with gr.Tabs():

        with gr.Tab('📷 Static Image'):
            gr.Markdown('Upload a face image. The model replicates it across 16 frames.')
            with gr.Row():
                with gr.Column(scale=1):
                    img_in  = gr.Image(label='Input Image', type='pil', height=280)
                    img_btn = gr.Button('🔍 Analyze', variant='primary')
                with gr.Column(scale=2):
                    img_label = gr.Textbox(label='Prediction')
                    img_probs = gr.Label(label='Confidence per Emotion', num_top_classes=7)
            with gr.Row():
                img_cam = gr.Image(label='GradCAM Attention Map', height=250)
                img_lm  = gr.Image(label='Geometry Landmark Overlay', height=250)
            img_btn.click(infer_image, inputs=[img_in],
                          outputs=[img_label, img_probs, img_cam, img_lm])

        with gr.Tab('🎬 Video Upload'):
            gr.Markdown('Upload a short video clip (MP4/AVI). 16 frames are sampled evenly.')
            with gr.Row():
                with gr.Column(scale=1):
                    vid_in  = gr.Video(label='Input Video', height=280)
                    vid_btn = gr.Button('🔍 Analyze', variant='primary')
                with gr.Column(scale=2):
                    vid_label = gr.Textbox(label='Prediction')
                    vid_probs = gr.Label(label='Confidence per Emotion', num_top_classes=7)
            with gr.Row():
                vid_cam = gr.Image(label='GradCAM Attention Map', height=250)
                vid_lm  = gr.Image(label='Geometry Landmark Overlay', height=250)
            vid_btn.click(infer_video, inputs=[vid_in],
                          outputs=[vid_label, vid_probs, vid_cam, vid_lm])

        with gr.Tab('📹 Webcam'):
            gr.Markdown('Capture a snapshot and analyze the expression.')
            with gr.Row():
                with gr.Column(scale=1):
                    cam_in  = gr.Image(sources=['webcam'], type='pil', label='Webcam', height=280)
                    cam_btn = gr.Button('🔍 Analyze', variant='primary')
                with gr.Column(scale=2):
                    cam_label = gr.Textbox(label='Prediction')
                    cam_probs = gr.Label(label='Confidence per Emotion', num_top_classes=7)
            with gr.Row():
                cam_cam = gr.Image(label='GradCAM Attention Map', height=250)
                cam_lm  = gr.Image(label='Geometry Landmark Overlay', height=250)
            cam_btn.click(infer_image, inputs=[cam_in],
                          outputs=[cam_label, cam_probs, cam_cam, cam_lm])

    gr.Markdown("""
    ---
    **Model:** GM-GReFEL trained on DFEW Fold 5 — **67.02% UAR / 77.14% WAR**  
    **Emotions:** Happy 😄 · Sad 😢 · Neutral 😐 · Angry 😠 · Surprise 😲 · Disgust 🤢 · Fear 😨  
    **GitHub:** [ardiytama/GroupMamba-DFER](https://github.com/ardiytama/GroupMamba-DFER)
    """)

demo.launch(share=True)